# AutoETF Research Agent MVP

面向行业 ETF 轮动的多因子研究、评分回测与权重诊断系统。

本 Notebook 只负责项目说明、调用 `src` 代码、展示结果和报告。正式运行前请先在 BigQuant 环境中执行字段探测。

In [ ]:
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "src").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f"project_root added: {p}")
        break

from src.agent_parser import parse_user_strategy
from src.factor_generator import generate_factor_candidates
from src.data_probe_bigquant import run_data_probe
from src.data_loader_bigquant import load_etf_data_bigquant
from src.factor_analysis import run_factor_analysis
from src.scoring import normalize_factors, build_weight_schemes, compute_composite_score
from src.backtest import run_top_pct_backtest
from src.diagnosis import diagnose_strategy
from src.report import generate_report, save_report
from src.plotting import plot_cumulative_return, plot_drawdown
from src.config import DEFAULT_CONFIG

config = DEFAULT_CONFIG

In [ ]:
def main():
    user_idea = "研究成交额放大、趋势走强、相对沪深300更强的行业ETF，未来是否更容易获得超额收益。"

    hypothesis = parse_user_strategy(user_idea)          # [AI-CORE]
    factors = generate_factor_candidates(hypothesis)     # [AI-CORE]

    # BigQuant 第一阶段必须先做字段探测。
    probe_results = run_data_probe()

    df = load_etf_data_bigquant(
        start_date=config.start_date,
        end_date=config.end_date,
        table_name=config.fund_table,
    )

    factor_cols = config.factor_cols
    target_cols = config.target_cols
    available_factor_cols = [col for col in factor_cols if col in df.columns]
    missing_factor_cols = [col for col in factor_cols if col not in df.columns]
    if missing_factor_cols:
        print(f"[WARN] missing factor columns skipped: {missing_factor_cols}")

    factor_directions = {
        factor["name"]: factor["direction"]
        for factor in factors
        if factor["name"] in available_factor_cols
    }

    factor_result = run_factor_analysis(df, available_factor_cols, target_cols)
    weight_schemes = build_weight_schemes(available_factor_cols, factor_result)
    weights = weight_schemes["hypothesis_weight"]

    scored_df = normalize_factors(df, factor_directions)
    scored_df = compute_composite_score(scored_df, weights)

    backtest_result = run_top_pct_backtest(
        scored_df,
        score_col="composite_score",
        target_col="future_return_5d",
        top_pct=config.top_pct,
        min_holdings=config.min_holdings,
        cost_bps=config.cost_bps,
    )

    diagnosis = diagnose_strategy(factor_result, backtest_result, weights)  # [AI-CORE]
    report = generate_report(hypothesis, factors, factor_result, backtest_result, diagnosis, weights)  # [AI-CORE]
    save_report(report)

    plot_cumulative_return(backtest_result["daily_returns"]["return"])
    plot_drawdown(backtest_result["daily_returns"]["return"])

    return {
        "hypothesis": hypothesis,
        "factors": factors,
        "probe_tables": list(probe_results.keys()),
        "data_shape": df.shape,
        "weight_schemes": weight_schemes,
        "backtest_performance": backtest_result["performance"],
        "diagnosis": diagnosis,
        "report": report,
    }

result = main()
result

## 工具调用示例

下面这个单元用变量输入模拟一次 Notebook 内智能体对话。DeepSeek 负责理解问题和总结结果，真实研究计算仍由 `src.agent_tools` 完成。

你可以不断修改 `user_input` 然后重复执行这一格，系统会把上一轮的 `state` 继续带下来。

In [ ]:
import getpass
import importlib

import src.notebook_chat_ui as notebook_chat_ui
importlib.reload(notebook_chat_ui)
from src.notebook_chat_ui import launch_notebook_chat
from src.secret_loader import load_deepseek_api_key

# 优先读取环境变量或项目根目录 .env；都没有时才手动输入。
api_key = load_deepseek_api_key(project_root)
if not api_key:
    api_key = getpass.getpass('DeepSeek API Key:')

chat_ui = launch_notebook_chat(
    config=config,
    use_llm=True,
    api_key=api_key,
    model='deepseek-v4-flash',
)


### 连续追问示例

这个单元演示如何保留上一轮上下文。你只要改 `user_input`，再运行一次，系统就会把条件、目标和部分结果状态带下去。

In [ ]:
# 连续对话示例：
# 1. 先运行上一个 cell，显示聊天框。
# 2. 然后直接在聊天框里输入新问题，不需要再手动调用 run_research_chat。
# 3. 例如：
#    - 加上成交额上升
#    - 改成未来10日
#    - 取消缩量，改成放量突破
